# Modern-model KV precision: K8/V4 versus K4/V8 at equal memory

**Run this experiment first.** It tests the practical key-precision finding on two public
checkpoints at longer contexts: **Qwen2.5-1.5B** and **SmolLM2-1.7B** (Llama architecture).
Both run without gated-model approval or a Hugging Face token.

1. In Colab, choose **Runtime → Change runtime type → A100 GPU** (40 or 80 GB).
2. Choose **Runtime → Run all**, and authorize the Drive mount.
3. Leave the defaults unchanged. The session budget is **10.5 hours**, including export.
4. Send back the final **`KV_results_*.zip`**. It contains the report, tables and all paired observations.

All code is embedded. The notebook uses an isolated Python environment. A timing pilot
estimates runtime before the main run; it never changes the scientific settings based
on results. Completed tables and partial evidence are saved to Drive throughout the run.

**Checkpoints:** [Qwen2.5-1.5B](https://huggingface.co/Qwen/Qwen2.5-1.5B) ·
[SmolLM2-1.7B](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B).

The implementation has been tested locally with small native Qwen2/Llama models.
The full GPU experiment runs when you execute this notebook.

## What is fixed before the run

| Item | Setting |
|---|---|
| Models | Qwen2.5-1.5B and SmolLM2-1.7B, immutable revisions |
| Context lengths | 1,024 / 2,048 / 4,096 tokens |
| Cache settings | K16/V16 reference; K8/V4; K4/V8 |
| Backbone | Original model weights and activations in BF16 |
| Measurements | 512 paired prefixes per model/context; 32 teacher-forced tokens per prefix |
| Data | Pinned WikiText-103 raw test token stream |
| Quantizer | Symmetric absmax per token/KV head; FP32 scales; packed 4-bit codes |
| Primary test | Exact strict-prefix-win probability; shared 5% error budget across six rows |
| Other results | Mean NLL gap, bootstrap interval, reference loss, prediction disagreement, actual bytes |

Every setting receives identical endpoints and target tokens. Context lengths also
share endpoints within each model. Keys are quantized after rotary embedding. The
code verifies equal packed cache bytes for K8/V4 and K4/V8, including scale tensors.

For each prefix, positive **NLL(K4/V8) − NLL(K8/V4)** favors key precision. The exact
primary test asks whether more than half the population of prefixes favor K8/V4.
Its one-sided confidence bounds are simultaneous across all six comparisons. Ties
count as non-wins; token positions are averaged within a prefix. The mean gap and
pointwise bootstrap interval show effect size separately. A complete report retains
all six comparisons, whichever allocation wins.

The scope is teacher-forced continuation quality for these checkpoints, quantizer,
and text population. Persistent KV storage is measured separately from peak GPU
memory and timing. Full protocol details and code are included in the result ZIP.

In [ ]:
#@title 1. Set up the experiment (keep defaults)
SAVE_TO_DRIVE = True #@param {type:"boolean"}
JOB_TAG = "" #@param {type:"string"}

import base64, hashlib, io, json, os, re, sys, zipfile
from pathlib import Path
from datetime import datetime, timezone

try:
    import google.colab
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab and select an A100 GPU.") from exc

if not JOB_TAG:
    JOB_TAG = "kv_precision_modern_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
if not re.fullmatch(r"kv_precision_modern_[A-Za-z0-9_-]{1,64}", JOB_TAG):
    raise ValueError("Leave JOB_TAG blank, or use kv_precision_modern_ followed by letters, digits, _ or -.")
BASE = Path("/content/KV_precision_modern_v1")
JOB = BASE / "jobs" / JOB_TAG
DRIVE_DIR = ""
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = str(Path("/content/drive/MyDrive/BCC_results") / JOB_TAG)
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

SOURCE_SHA = "aeb4a7a39cff5c91f5a15ed41e2cf55f8b020360c3b333e53b9e9bd8724058b1"
EMBEDDED_SOURCE = "..."
payload = base64.b64decode(EMBEDDED_SOURCE)
if hashlib.sha256(payload).hexdigest() != SOURCE_SHA:
    raise RuntimeError("Embedded source checksum mismatch.")
CODE = BASE / ("code_" + SOURCE_SHA[:12])
CODE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for item in archive.infolist():
        target = (CODE / item.filename).resolve()
        if not target.is_relative_to(CODE.resolve()):
            raise RuntimeError("Invalid embedded archive path.")
        content = archive.read(item)
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and target.read_bytes() != content:
            raise RuntimeError("Extracted source was changed. Start a fresh runtime.")
        if not target.exists():
            target.write_bytes(content)
JOB.mkdir(parents=True, exist_ok=True)
config = json.loads((CODE / "protocol.json").read_text())
print("Models:", [m["id"] for m in config["models"]])
print("Contexts:", config["context_lengths"], "| prefixes:", config["prefixes_per_comparison"])
print("Job:", JOB)
print("Drive folder:", DRIVE_DIR or "disabled; download before the runtime is deleted")
print("Source SHA256:", SOURCE_SHA)


In [ ]:
#@title 2. Run the experiment (up to 10.5 hours)
import subprocess, signal
if globals().get("SOURCE_SHA") != "aeb4a7a39cff5c91f5a15ed41e2cf55f8b020360c3b333e53b9e9bd8724058b1":
    raise RuntimeError("Run cell 1 from this notebook first.")
cmd = [sys.executable, str(CODE / "controller.py"), "--job", str(JOB),
       "--budget-hours", "10.5", "--drive-dir", DRIVE_DIR]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    print("Stopping computation and saving available results. Please wait.")
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
print("Controller exit code:", returncode)
print("Run the next cell to display the report and download your ZIP.")


## Retrieve the report and results

This cell downloads the compact **`KV_results_*.zip`**. The same file is saved under
**Drive → MyDrive → BCC_results → kv_precision_modern_…**.

**After a runtime reset:** run this cell alone. It mounts Drive and retrieves the
latest saved run. Enter a particular run folder in `RESULT_FOLDER` to select it.
If a run ended before final export, it retrieves the last checkpoint and shows its
completion status. All observations needed to rebuild the tables are inside the ZIP.

In [ ]:
#@title 3. Show the report and download the ZIP (also works after a runtime reset)
RESULT_FOLDER = "" #@param {type:"string"}

from pathlib import Path
import hashlib, json, zipfile
from google.colab import drive, files
from IPython.display import display, Markdown

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(4*1024*1024), b""):
            h.update(block)
    return h.hexdigest()

folders = []
if RESULT_FOLDER.strip():
    folders = [Path(RESULT_FOLDER.strip())]
else:
    if globals().get("JOB") is not None:
        folders.append(Path(JOB) / "exports")
    if globals().get("DRIVE_DIR"):
        folders.append(Path(DRIVE_DIR))
if not folders or not any(p.is_dir() and list(p.glob("KV_*.zip")) for p in folders):
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    if RESULT_FOLDER.strip():
        folders = [Path(RESULT_FOLDER.strip())]
    else:
        folders += sorted(Path("/content/drive/MyDrive/BCC_results").glob("kv_precision_modern_*"), reverse=True)

archive = None
for folder in folders:
    for directory in [folder, folder / "exports"]:
        candidates = sorted(directory.glob("KV_results_*.zip"), reverse=True)
        if not candidates:
            candidates = sorted(directory.glob("KV_checkpoint_*.zip"), reverse=True)
        if candidates:
            archive = candidates[0]
            break
    if archive is not None:
        break
if archive is None:
    raise FileNotFoundError("No saved KV precision ZIP found. Check your Drive BCC_results folder or run cell 2.")

with zipfile.ZipFile(archive) as result:
    bad = result.testzip()
    if bad is not None:
        raise RuntimeError("Archive CRC failed: " + bad)
    manifest = json.loads(result.read("MANIFEST.json"))
    for entry in manifest:
        data = result.read(entry["path"])
        if len(data) != entry["bytes"] or hashlib.sha256(data).hexdigest() != entry["sha256"]:
            raise RuntimeError("Manifest check failed: " + entry["path"])
    if "STATUS.json" in result.namelist():
        print(result.read("STATUS.json").decode())
    if "REPORT.md" in result.namelist():
        display(Markdown(result.read("REPORT.md").decode()))
    else:
        print("This archive contains an incomplete run. Its logs and frozen inputs are preserved.")
print("Verified archive:", archive)
print("Size:", round(archive.stat().st_size/2**20, 2), "MiB")
print("SHA256:", file_sha256(archive))
print("If browser download is blocked, download this same ZIP from your Drive BCC_results folder.")
files.download(str(archive))
